# Brain Tumor Detection from MRI (Tumor vs No Tumor)

This notebook trains a deep learning model that looks at a brain MRI scan and says whether a tumor is present.

**What it produces at the end** (one zip file, `project_outputs.zip`):
- trained model weights (`.keras` and `.onnx`)
- all result charts (confusion matrix, ROC curve, training curves, Grad-CAM heatmaps)
- the model and sample scans for the Vercel website
- `project.json` with the real test scores, which the website reads automatically

**How to run it**
1. In Colab, go to *Runtime > Change runtime type* and pick **T4 GPU**.
2. Click *Runtime > Run all*. The full run takes about 30 to 40 minutes.
3. At the end, `project_outputs.zip` downloads. Unzip it into your GitHub repo folder.

Tip: set `QUICK_TEST = True` in Step 1 first. It runs every step on a tiny slice of data in a few minutes, so you can catch problems before the long run.

## Step 0: Install the few extra tools
Colab already has TensorFlow. We add `kagglehub` (downloads the dataset), and `tf2onnx` + `onnxruntime` (to export the model so it can run inside a web browser).

In [ ]:
!pip -q install kagglehub tf2onnx onnx onnxruntime

## Step 1: Settings
Everything you might want to change is in this one cell.
- `TARGET_RECALL = 0.98` means: tune the model so it catches at least 98% of tumors on the validation set. In medical screening, missing a tumor is far worse than a false alarm, so we favor catching tumors.

In [ ]:
import os, json, time, random, hashlib, shutil, zipfile, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, roc_curve, roc_auc_score,
                             accuracy_score, precision_score, recall_score, f1_score)
warnings.filterwarnings('ignore')

QUICK_TEST    = False   # True = tiny smoke test (a few minutes). False = real training.
RUN_BASELINES = True    # also train the two models from the original project, for comparison
SEED          = 42
IMG_SIZE      = 224
BATCH         = 32
TARGET_RECALL = 0.98
EPOCHS = dict(cnn=25, vgg=12, head=8, finetune=15)
if QUICK_TEST:
    EPOCHS = dict(cnn=1, vgg=1, head=1, finetune=1)

OUT = 'project_outputs'
for d in ['weights', 'results', 'web/model', 'web/samples', 'web/results']:
    os.makedirs(os.path.join(OUT, d), exist_ok=True)

keras.utils.set_random_seed(SEED)
print('TensorFlow', tf.__version__, '| Keras', keras.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPU:', gpus if gpus else 'NONE. Training will be very slow. Switch the runtime to T4 GPU.')

## Step 2: Download the dataset
Brain Tumor MRI Dataset (Masoud Nickparvar, Kaggle). Four folders: glioma, meningioma, pituitary, notumor.
We turn it into a two-class problem: the three tumor types become **tumor** (1), and notumor stays **no tumor** (0).
We do not copy any files. We just build a table of file paths and labels, which is faster and keeps the original data untouched.

In [ ]:
import kagglehub
DATASET_SLUG = 'masoudnickparvar/brain-tumor-mri-dataset'
DATA_ROOT = kagglehub.dataset_download(DATASET_SLUG)
print('Downloaded to:', DATA_ROOT)

def find_dir(root, name):
    for dirpath, dirnames, _ in os.walk(root):
        for d in dirnames:
            if d.lower() == name.lower():
                return os.path.join(dirpath, d)
    raise FileNotFoundError(f'No folder called {name} under {root}')

def is_no_tumor(folder_name):
    key = folder_name.lower().replace('_', '').replace(' ', '').replace('-', '')
    return key in ('notumor', 'no', 'healthy', 'normal')

rows = []
for split in ['Training', 'Testing']:
    split_dir = find_dir(DATA_ROOT, split)
    for cls in sorted(os.listdir(split_dir)):
        cls_dir = os.path.join(split_dir, cls)
        if not os.path.isdir(cls_dir):
            continue
        for f in sorted(os.listdir(cls_dir)):
            if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                rows.append(dict(path=os.path.join(cls_dir, f), split=split.lower(),
                                 subtype=cls.lower(), label=0 if is_no_tumor(cls) else 1))
df = pd.DataFrame(rows)
print(df.groupby(['split', 'subtype']).size().unstack(0))

## Step 3: Remove duplicate images
Public datasets often contain the same scan more than once. If a copy lands in training and another copy in validation or test, the model is being tested on images it has already seen, and the scores look better than they really are.
We fingerprint every file (MD5 hash), drop any test image that also exists in training, and keep only one copy of each image inside a split.

In [ ]:
df['md5'] = df['path'].map(lambda p: hashlib.md5(open(p, 'rb').read()).hexdigest())
n_before = len(df)
train_hashes = set(df.loc[df.split == 'training', 'md5'])
leak = (df.split == 'testing') & df.md5.isin(train_hashes)
df = df[~leak].drop_duplicates(['split', 'md5']).reset_index(drop=True)
DUPES_REMOVED = n_before - len(df)
print(f'Removed {DUPES_REMOVED} duplicate images ({int(leak.sum())} of them were test images also present in training).')
print('Images left:', len(df))

## Step 4: Split into train / validation / test
- **Test** = the dataset's own Testing folder. We touch it only once, at the very end.
- **Validation** = 15% of the Training folder. Used to stop training at the right time and to pick the decision threshold.
- **Train** = the remaining 85%.

The original project used the test set for validation, which quietly leaks test information into training decisions. Keeping them separate makes the final score honest.

In [ ]:
train_full = df[df.split == 'training']
test_df    = df[df.split == 'testing'].reset_index(drop=True)
train_df, val_df = train_test_split(train_full, test_size=0.15, stratify=train_full.subtype,
                                    random_state=SEED)
train_df, val_df = train_df.reset_index(drop=True), val_df.reset_index(drop=True)

if QUICK_TEST:
    train_df = train_df.groupby('subtype').head(40).reset_index(drop=True)
    val_df   = val_df.groupby('subtype').head(12).reset_index(drop=True)
    test_df  = test_df.groupby('subtype').head(12).reset_index(drop=True)

summary = pd.DataFrame({name: frame.subtype.value_counts() for name, frame in
                        [('train', train_df), ('val', val_df), ('test', test_df)]})
summary.loc['TOTAL'] = summary.sum()
print(summary)
for name, frame in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print(f'{name:5s}  tumor={int(frame.label.sum()):5d}  no tumor={int((1 - frame.label).sum()):5d}')

## Step 5: Look at the data
A quick chart of how many images each class has, and a few example scans.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
summary.drop('TOTAL').plot.bar(ax=ax[0], color=['#1f4e79', '#7fa7c9', '#c9d9e8'], edgecolor='black')
ax[0].set_title('Images per subtype and split'); ax[0].set_ylabel('Images'); ax[0].tick_params(axis='x', rotation=0)
binary = pd.DataFrame({n: [int(f.label.sum()), int((1 - f.label).sum())] for n, f in
                       [('train', train_df), ('val', val_df), ('test', test_df)]}, index=['tumor', 'no tumor'])
binary.T.plot.bar(ax=ax[1], color=['#b42345', '#0f7b6c'], edgecolor='black')
ax[1].set_title('Tumor vs no tumor (about 3 to 1)'); ax[1].tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.savefig(f'{OUT}/results/class_distribution.png', dpi=150); plt.show()

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for col, sub in enumerate(['glioma', 'meningioma', 'pituitary', 'notumor']):
    pick = train_df[train_df.subtype == sub].sample(2, random_state=SEED)
    for row, p in enumerate(pick.path):
        axes[row, col].imshow(plt.imread(p), cmap='gray'); axes[row, col].axis('off')
        axes[row, col].set_title(sub, color='#0f7b6c' if sub == 'notumor' else '#b42345')
plt.suptitle('Example MRI scans'); plt.tight_layout()
plt.savefig(f'{OUT}/results/sample_images.png', dpi=150); plt.show()

## Step 6: Input pipeline and data augmentation
Every image is resized to 224 x 224 and kept as normal pixel values (0 to 255). **The model does its own scaling inside**, so the Python script, the website and the automated evaluator only need to resize the image. That removes the most common deployment bug (preprocessing done differently in training and in the app).

Augmentation (small flips, rotations, zooms, contrast changes) is applied only to training images, so the model sees more variety and overfits less.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def load_image(path, label):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE), antialias=True)
    return tf.cast(tf.clip_by_value(tf.round(img), 0, 255), tf.uint8), label   # uint8 keeps the cache small

augment = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.10),
    layers.RandomTranslation(0.05, 0.05),
    layers.RandomContrast(0.15),
], name='augmentation')

def make_ds(frame, training):
    ds = tf.data.Dataset.from_tensor_slices((frame.path.values, frame.label.values.astype('float32')))
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE).cache()
    if training:
        ds = ds.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(BATCH).map(lambda x, y: (tf.cast(x, tf.float32), y), num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

train_ds, val_ds, test_ds = make_ds(train_df, True), make_ds(val_df, False), make_ds(test_df, False)

# Tumor images outnumber no-tumor images about 3 to 1. Class weights stop the model from just favoring 'tumor'.
n_pos, n_neg = int(train_df.label.sum()), int((1 - train_df.label).sum())
CLASS_WEIGHT = {0: (n_pos + n_neg) / (2 * n_neg), 1: (n_pos + n_neg) / (2 * n_pos)}
print('Class weights:', {k: round(v, 3) for k, v in CLASS_WEIGHT.items()})

## Step 7: Training helpers
Shared settings for every model:
- **AUC, recall, precision** are tracked, not just accuracy, because accuracy hides missed tumors.
- **Early stopping** watches validation AUC and keeps the best epoch.
- **Learning-rate reduction** slows learning down when progress stalls.

In [ ]:
def compile_model(model, lr):
    model.compile(optimizer=keras.optimizers.Adam(lr), loss='binary_crossentropy',
                  metrics=[keras.metrics.BinaryAccuracy(name='accuracy'), keras.metrics.AUC(name='auc'),
                           keras.metrics.Recall(name='recall'), keras.metrics.Precision(name='precision')])

def callbacks(patience=4):
    return [keras.callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=patience,
                                          restore_best_weights=True, verbose=1),
            keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2,
                                              min_lr=1e-7, verbose=1)]

def fit(model, epochs, patience=4):
    t = time.time()
    h = model.fit(train_ds, validation_data=val_ds, epochs=epochs, class_weight=CLASS_WEIGHT,
                  callbacks=callbacks(patience), verbose=1)
    print(f'Training took {(time.time() - t) / 60:.1f} min')
    return h.history

def predict_probs(model, ds):
    return model.predict(ds, verbose=0).ravel()

HISTORY, PROBS = {}, {}

## Step 8 (baseline 1): Custom CNN from the original project
Same four-block idea as the original, with one upgrade: Global Average Pooling instead of Flatten. Flatten created millions of extra weights and was the main reason the original CNN overfitted.

In [ ]:
def build_custom_cnn():
    inputs = keras.Input((IMG_SIZE, IMG_SIZE, 3), name='image')
    x = layers.Rescaling(1 / 255.0)(inputs)
    for filters in [32, 64, 128, 256]:
        x = layers.Conv2D(filters, 3, padding='same', use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.MaxPooling2D()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(1, activation='sigmoid', name='tumor_probability')(x)
    return keras.Model(inputs, outputs, name='custom_cnn')

if RUN_BASELINES:
    cnn = build_custom_cnn()
    compile_model(cnn, 1e-3)
    HISTORY['Custom CNN'] = fit(cnn, EPOCHS['cnn'], patience=6)
    PROBS['Custom CNN'] = (predict_probs(cnn, val_ds), predict_probs(cnn, test_ds))
    del cnn; keras.backend.clear_session()

## Step 9 (baseline 2): VGG16 transfer learning from the original project
VGG16 pretrained on ImageNet with its layers frozen and a small new head on top, as in the original project.

In [ ]:
def build_vgg16():
    base = keras.applications.VGG16(include_top=False, weights='imagenet', input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False
    inputs = keras.Input((IMG_SIZE, IMG_SIZE, 3), name='image')
    x = keras.applications.vgg16.preprocess_input(inputs)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(1, activation='sigmoid', name='tumor_probability')(x)
    return keras.Model(inputs, outputs, name='vgg16_transfer')

if RUN_BASELINES:
    vgg = build_vgg16()
    compile_model(vgg, 1e-3)
    HISTORY['VGG16 (frozen)'] = fit(vgg, EPOCHS['vgg'])
    PROBS['VGG16 (frozen)'] = (predict_probs(vgg, val_ds), predict_probs(vgg, test_ds))
    del vgg; keras.backend.clear_session()

## Step 10 (main model): EfficientNetB0 with fine-tuning
This is the model we deploy. EfficientNetB0 is more accurate than VGG16 while being about 4 times smaller (around 16 MB), so it loads quickly in a web browser and fits easily in GitHub.

Training happens in two phases:
1. **Head only.** The pretrained network is frozen and only the new top layer learns.
2. **Fine-tuning.** The whole network is unlocked and trained with a very small learning rate, so it adapts its image features to MRI scans without forgetting what it already knows. BatchNorm layers stay frozen, which is the standard way to keep fine-tuning stable.

In [ ]:
def build_effnet(weights='imagenet'):
    base = keras.applications.EfficientNetB0(include_top=False, weights=weights,
                                             input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False
    inputs = keras.Input((IMG_SIZE, IMG_SIZE, 3), name='image')   # raw RGB pixels 0..255
    x = base(inputs, training=False)                                # scaling happens inside EfficientNet
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.Dropout(0.3, name='dropout')(x)
    outputs = layers.Dense(1, activation='sigmoid', name='tumor_probability')(x)
    return keras.Model(inputs, outputs, name='effnetb0_brain_tumor'), base

model, base = build_effnet()

# Phase 1: train the head
compile_model(model, 1e-3)
h1 = fit(model, EPOCHS['head'], patience=3)

# Phase 2: fine-tune everything except BatchNorm
base.trainable = True
for layer in base.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False
compile_model(model, 3e-5)
h2 = fit(model, EPOCHS['finetune'], patience=5)

HISTORY['EfficientNetB0 (fine-tuned)'] = {k: h1[k] + h2.get(k, []) for k in h1}
HEAD_EPOCHS = len(h1['loss'])
PROBS['EfficientNetB0 (fine-tuned)'] = (predict_probs(model, val_ds), predict_probs(model, test_ds))

## Step 11: Pick the decision threshold (on validation data only)
The model outputs a probability from 0 to 1. The usual cut-off is 0.5, but for screening we want to miss as few tumors as possible.
We find the highest threshold that still catches at least `TARGET_RECALL` of tumors on the **validation** set, and never go above 0.5. The test set is not used here, so the final score stays honest.

In [ ]:
def pick_threshold(y_true, probs, target=TARGET_RECALL):
    fpr, tpr, thr = roc_curve(y_true, probs)
    idx = np.where(tpr >= target)[0]
    t = float(thr[idx[0]]) if len(idx) else 0.5
    return float(min(0.5, max(t, 1e-4)))

def evaluate(y_true, probs, threshold):
    pred = (probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return dict(threshold=round(threshold, 4),
                accuracy=accuracy_score(y_true, pred),
                sensitivity=recall_score(y_true, pred, zero_division=0),      # tumors caught
                specificity=tn / (tn + fp) if (tn + fp) else 0.0,             # healthy scans cleared
                precision=precision_score(y_true, pred, zero_division=0),
                f1=f1_score(y_true, pred, zero_division=0),
                auc=roc_auc_score(y_true, probs),
                tp=int(tp), fn=int(fn), tn=int(tn), fp=int(fp))

y_val, y_test = val_df.label.values, test_df.label.values
RESULTS, THRESHOLDS = {}, {}
for name, (p_val, p_test) in PROBS.items():
    THRESHOLDS[name] = pick_threshold(y_val, p_val)
    RESULTS[name] = evaluate(y_test, p_test, THRESHOLDS[name])
    RESULTS[name + ' @0.5'] = evaluate(y_test, p_test, 0.5)

table = pd.DataFrame(RESULTS).T
show = table[['threshold', 'accuracy', 'sensitivity', 'specificity', 'precision', 'f1', 'auc', 'fn', 'fp']].copy()
for c in ['accuracy', 'sensitivity', 'specificity', 'precision', 'f1', 'auc']:
    show[c] = (show[c].astype(float) * 100).round(2)
print('TEST SET RESULTS (percent; fn = missed tumors, fp = false alarms)')
print(show.to_string())
show.to_csv(f'{OUT}/results/model_comparison.csv')

MAIN = 'EfficientNetB0 (fine-tuned)'
THRESHOLD = THRESHOLDS[MAIN]
FINAL = RESULTS[MAIN]
print(f'\nDeployed model: {MAIN}  |  threshold = {THRESHOLD:.4f}')

## Step 12: Training curves, confusion matrix and ROC curve

In [ ]:
h = HISTORY[MAIN]
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
for i, (k, title) in enumerate([('loss', 'Loss'), ('accuracy', 'Accuracy'), ('auc', 'AUC')]):
    ax[i].plot(h[k], label='train', color='#1f4e79'); ax[i].plot(h['val_' + k], label='validation', color='#b42345')
    ax[i].axvline(HEAD_EPOCHS - 0.5, ls='--', color='gray'); ax[i].set_title(f'{title} (dashed line = fine-tuning starts)')
    ax[i].set_xlabel('Epoch'); ax[i].legend()
plt.tight_layout(); plt.savefig(f'{OUT}/results/training_curves.png', dpi=150); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
cm = np.array([[FINAL['tn'], FINAL['fp']], [FINAL['fn'], FINAL['tp']]])
ax[0].imshow(cm, cmap='Blues')
for (r, c), v in np.ndenumerate(cm):
    ax[0].text(c, r, v, ha='center', va='center', fontsize=18, color='white' if v > cm.max() / 2 else 'black')
ax[0].set_xticks([0, 1], ['No tumor', 'Tumor']); ax[0].set_yticks([0, 1], ['No tumor', 'Tumor'])
ax[0].set_xlabel('Predicted'); ax[0].set_ylabel('Actual'); ax[0].set_title(f'Confusion matrix (test, threshold {THRESHOLD:.3f})')
for name, (_, p_test) in PROBS.items():
    fpr, tpr, _ = roc_curve(y_test, p_test)
    ax[1].plot(fpr, tpr, label=f'{name} (AUC {roc_auc_score(y_test, p_test):.4f})')
ax[1].plot([0, 1], [0, 1], ls='--', color='gray'); ax[1].set_xlabel('False positive rate')
ax[1].set_ylabel('True positive rate (sensitivity)'); ax[1].set_title('ROC curve (test set)'); ax[1].legend(loc='lower right')
plt.tight_layout(); plt.savefig(f'{OUT}/results/confusion_and_roc.png', dpi=150); plt.show()

# Which tumor types get missed? Useful for the report.
p_main = PROBS[MAIN][1]
per_type = test_df.assign(pred=(p_main >= THRESHOLD).astype(int)).groupby('subtype').apply(
    lambda g: pd.Series({'images': len(g), 'correct': int((g.pred == g.label).sum())}))
per_type['correct %'] = (per_type.correct / per_type.images * 100).round(2)
print(per_type)
per_type.to_csv(f'{OUT}/results/per_subtype.csv')

## Step 13: Grad-CAM, where is the model looking?
Grad-CAM colors the parts of the scan that pushed the model toward its answer. If the hot spots sit on the tumor area (and not on the skull edge or text labels), that is good evidence the model learned something real. These images are great for the report, slides and website.

In [ ]:
def grad_cam(img):
    x = tf.convert_to_tensor(img[None], tf.float32)
    with tf.GradientTape() as tape:
        conv = base(x, training=False)
        tape.watch(conv)
        y = model.get_layer('tumor_probability')(model.get_layer('dropout')(model.get_layer('gap')(conv), training=False))
    grads = tape.gradient(y, conv)[0]
    cam = tf.nn.relu(tf.reduce_sum(conv[0] * tf.reduce_mean(grads, axis=(0, 1)), axis=-1))
    cam = cam / (tf.reduce_max(cam) + 1e-8)
    return tf.image.resize(cam[..., None], (IMG_SIZE, IMG_SIZE)).numpy()[..., 0], float(y[0, 0])

def load_rgb(path):
    return load_image(path, 0)[0].numpy().astype('float32')

picks = pd.concat([test_df[test_df.subtype == s].sample(2, random_state=SEED)
                   for s in ['glioma', 'meningioma', 'pituitary', 'notumor']])
fig, axes = plt.subplots(2, 8, figsize=(22, 6))
for i, (_, r) in enumerate(picks.iterrows()):
    img = load_rgb(r.path); cam, p = grad_cam(img)
    axes[0, i].imshow(img.astype('uint8')); axes[0, i].set_title(r.subtype); axes[0, i].axis('off')
    axes[1, i].imshow(img.astype('uint8')); axes[1, i].imshow(cam, cmap='jet', alpha=0.4)
    axes[1, i].set_title(f'tumor prob {p:.2f}'); axes[1, i].axis('off')
plt.suptitle('Grad-CAM: top row original, bottom row where the model looked'); plt.tight_layout()
plt.savefig(f'{OUT}/results/gradcam.png', dpi=130); plt.show()

# The mistakes, so you can discuss them honestly in the report
wrong = test_df.assign(prob=p_main)[(p_main >= THRESHOLD).astype(int) != test_df.label].sort_values('prob')
print(f'{len(wrong)} test images misclassified')
if len(wrong):
    show_wrong = wrong.head(4) if len(wrong) <= 8 else pd.concat([wrong.head(4), wrong.tail(4)])
    fig, axes = plt.subplots(1, len(show_wrong), figsize=(3 * len(show_wrong), 3.4), squeeze=False)
    for ax, (_, r) in zip(axes[0], show_wrong.iterrows()):
        ax.imshow(plt.imread(r.path), cmap='gray'); ax.axis('off')
        ax.set_title(f'{r.subtype}\nprob {r.prob:.2f}', fontsize=9)
    plt.tight_layout(); plt.savefig(f'{OUT}/results/mistakes.png', dpi=130); plt.show()

## Step 14: Save the model (Keras + ONNX) and check both give the same answers
- `brain_tumor_effnetb0.keras` is the normal TensorFlow model file (about 17 MB, no optimizer state, so it stays small).
- `brain_tumor_effnetb0.onnx` is the same model in a format that runs inside a web browser (used by the Vercel site) and in `predict.py` without installing TensorFlow.

The last part of this cell runs both files on 32 test images and confirms the outputs match. Never ship a model you have not reloaded and checked.

In [ ]:
import onnx, onnxruntime as ort, tf2onnx
from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2

KERAS_PATH = f'{OUT}/weights/brain_tumor_effnetb0.keras'
ONNX_PATH  = f'{OUT}/weights/brain_tumor_effnetb0.onnx'

# Save once, reload without the optimizer, save again: removes about 30 MB of training-only state.
model.save('/tmp/full_model.keras')
deploy = keras.models.load_model('/tmp/full_model.keras', compile=False)
deploy.save(KERAS_PATH)

# Freeze the graph and convert to ONNX (this route keeps EfficientNet's built-in normalization inside the file)
spec = tf.TensorSpec([None, IMG_SIZE, IMG_SIZE, 3], tf.float32, name='image')
concrete = tf.function(lambda image: deploy(image, training=False)).get_concrete_function(spec)
frozen = convert_variables_to_constants_v2(concrete)
proto, _ = tf2onnx.convert.from_graph_def(frozen.graph.as_graph_def(),
                                          input_names=[t.name for t in frozen.inputs],
                                          output_names=[t.name for t in frozen.outputs], opset=17)

# Give the input and output simple names: 'image' and 'tumor_probability'
rename = {proto.graph.input[0].name: 'image', proto.graph.output[0].name: 'tumor_probability'}
for v in list(proto.graph.input) + list(proto.graph.output):
    v.name = rename.get(v.name, v.name)
for node in proto.graph.node:
    node.input[:] = [rename.get(n, n) for n in node.input]
    node.output[:] = [rename.get(n, n) for n in node.output]
onnx.checker.check_model(proto)
onnx.save(proto, ONNX_PATH)

# Parity check: reloaded Keras file vs ONNX file vs the trained model in memory
check = np.stack([load_rgb(p) for p in test_df.path.values[:32]])
p_mem  = model.predict(check, verbose=0).ravel()
p_k    = keras.models.load_model(KERAS_PATH).predict(check, verbose=0).ravel()
sess   = ort.InferenceSession(ONNX_PATH)
p_onnx = sess.run(None, {'image': check})[0].ravel()
print('inputs :', [(i.name, i.shape) for i in sess.get_inputs()])
print('outputs:', [(o.name, o.shape) for o in sess.get_outputs()])
print(f'max difference  keras file: {np.abs(p_k - p_mem).max():.2e}   onnx file: {np.abs(p_onnx - p_mem).max():.2e}')
assert np.abs(p_onnx - p_mem).max() < 1e-3 and np.abs(p_k - p_mem).max() < 1e-4, 'Exported model does not match!'
for f in [KERAS_PATH, ONNX_PATH]:
    print(f'{os.path.basename(f):32s} {os.path.getsize(f) / 1e6:6.1f} MB')
shutil.copy(ONNX_PATH, f'{OUT}/web/model/brain_tumor_effnetb0.onnx')
print('Saved and verified.')

## Step 15: Write the config, the website data and sample scans
- `weights/config.json` tells `predict.py` (and the automated evaluator) how to use the model: input size, threshold, labels.
- `web/project.json` holds the real test scores. The website reads it, so you never type numbers by hand.
- Six test scans (never seen in training) are copied into `web/samples/` so visitors can try the model with one click.

In [ ]:
fmt = lambda v: round(float(v), 4)
MODEL_INFO = {
    'name': 'EfficientNetB0 fine-tuned (binary)',
    'task': 'Brain MRI: tumor vs no tumor',
    'files': {'keras': 'weights/brain_tumor_effnetb0.keras', 'onnx': 'weights/brain_tumor_effnetb0.onnx'},
    'input': {'name': 'image', 'shape': [None, IMG_SIZE, IMG_SIZE, 3], 'dtype': 'float32',
              'format': 'RGB pixels 0-255, resized to 224x224, no normalization (the model scales internally)'},
    'output': {'name': 'tumor_probability', 'meaning': 'probability that the scan contains a tumor'},
    'threshold': fmt(THRESHOLD),
    'threshold_note': f'Chosen on the validation set to catch at least {int(TARGET_RECALL * 100)}% of tumors (capped at 0.5)',
    'labels': {'0': 'No tumor', '1': 'Tumor'},
}
with open(f'{OUT}/weights/config.json', 'w') as f:
    json.dump(MODEL_INFO, f, indent=2)

samples = []
rng = np.random.default_rng(SEED)
for sub, n in [('glioma', 1), ('meningioma', 1), ('pituitary', 1), ('notumor', 3)]:
    pool = test_df[test_df.subtype == sub]
    for path in pool.path.values[rng.choice(len(pool), n, replace=False)]:
        name = f'{sub}_{len(samples) + 1}.jpg'
        from PIL import Image
        im = Image.open(path).convert('RGB'); im.thumbnail((512, 512)); im.save(f'{OUT}/web/samples/{name}', quality=90)
        samples.append({'file': f'samples/{name}', 'truth': 'No tumor' if sub == 'notumor' else 'Tumor', 'subtype': sub})

comparison = [{'model': k, 'threshold': fmt(v['threshold']), 'accuracy': fmt(v['accuracy']),
               'sensitivity': fmt(v['sensitivity']), 'specificity': fmt(v['specificity']),
               'precision': fmt(v['precision']), 'f1': fmt(v['f1']), 'auc': fmt(v['auc']),
               'missed_tumors': v['fn'], 'false_alarms': v['fp']}
              for k, v in RESULTS.items() if not k.endswith('@0.5')]

project = {
    'title': 'Brain tumor detection from MRI',
    'generated': time.strftime('%Y-%m-%d %H:%M UTC', time.gmtime()),
    'quick_test': QUICK_TEST,
    'model': {**MODEL_INFO, 'web_file': 'model/brain_tumor_effnetb0.onnx'},
    'test_metrics': {k: (fmt(v) if isinstance(v, float) else v) for k, v in FINAL.items()},
    'comparison': comparison,
    'dataset': {'name': 'Brain Tumor MRI Dataset (Masoud Nickparvar)',
                'url': f'https://www.kaggle.com/datasets/{DATASET_SLUG}',
                'duplicates_removed': int(DUPES_REMOVED),
                'train': len(train_df), 'validation': len(val_df), 'test': len(test_df),
                'classes': 'glioma + meningioma + pituitary = Tumor; notumor = No tumor'},
    'training': {'image_size': IMG_SIZE, 'batch_size': BATCH, 'head_epochs': HEAD_EPOCHS,
                 'total_epochs': len(HISTORY[MAIN]['loss']), 'optimizer': 'Adam (1e-3 head, 3e-5 fine-tune)',
                 'framework': f'TensorFlow {tf.__version__} / Keras {keras.__version__}'},
    'samples': samples,
}
with open(f'{OUT}/web/project.json', 'w') as f:
    json.dump(project, f, indent=2)

for img in ['confusion_and_roc.png', 'gradcam.png', 'training_curves.png', 'class_distribution.png']:
    shutil.copy(f'{OUT}/results/{img}', f'{OUT}/web/results/{img}')
pd.DataFrame(HISTORY[MAIN]).to_csv(f'{OUT}/results/training_history.csv', index_label='epoch')

with open(f'{OUT}/results/metrics.md', 'w') as f:
    f.write('| Model | Threshold | Accuracy | Sensitivity | Specificity | Precision | F1 | ROC-AUC | Missed tumors | False alarms |\n')
    f.write('|---|---|---|---|---|---|---|---|---|---|\n')
    for c in comparison:
        f.write(f"| {c['model']} | {c['threshold']} | {c['accuracy']*100:.2f}% | {c['sensitivity']*100:.2f}% | "
                f"{c['specificity']*100:.2f}% | {c['precision']*100:.2f}% | {c['f1']*100:.2f}% | {c['auc']:.4f} | "
                f"{c['missed_tumors']} | {c['false_alarms']} |\n")
print(open(f'{OUT}/results/metrics.md').read())

## Step 16: Download everything
Unzip `project_outputs.zip` into the root of your GitHub repo folder. It fills `weights/`, `results/`, `web/model/`, `web/samples/`, `web/results/` and `web/project.json`.

In [ ]:
zip_path = 'project_outputs.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(OUT):
        for f in files:
            full = os.path.join(root, f)
            z.write(full, os.path.relpath(full, OUT))
print(f'{zip_path}: {os.path.getsize(zip_path) / 1e6:.1f} MB')
if QUICK_TEST:
    print('QUICK_TEST is on. These are NOT real results. Set QUICK_TEST = False and run all again.')
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print('Not running in Colab. The zip is in the current folder.')